In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import style
import re
import math
import copy

In [ ]:
def preProc(dataFileInput):
  
  df = 'https://raw.githubusercontent.com/rocktrees/assignment3/main/reuters_health.txt'

  df = pd.read_csv(dataFileInput,header = None, sep = "|")

  df = df.sample(frac = 1)

  df = df.drop(df.columns[[0,1]], axis = 1)

  df = df.dropna(axis = 0, how = 'any')

  df.columns = ['Input']

  df = df['Input'].apply(tweetFix)

  print(df)

  print("\n")
 
  return df

In [ ]:
def tweetFix(tweet):

  df = 'https://raw.githubusercontent.com/rocktrees/assignment3/main/reuters_health.txt'

  rem_func = lambda x: x[0] != '@' and x[:7] != 'http://'

  rem = filter(rem_func, tweet.split())

  newRem = []

  for word in rem:

    if word[0] == "#":

      newWord = word[1:]

      newRem.append(newWord[1:].lower().strip("'"))

    else:

      newRem.append(word.lower().strip("'"))

  
  return newRem

In [ ]:
def clump(clustNum, dataFileInput, centro):

  i = 0
  
  clump = {}

  while(i < clustNum):

    clump[i] = []

    i = i + 1

  for tweets in dataFileInput:

    distTweet = []

    for point in centro:

      temp = distJaccard(tweets, centro[point])

      distTweet.append(temp)


    minimumVal = min(distTweet)

    minimum_dist = distTweet.index(minimumVal)
    
    clump[minimum_dist].append(tweets)
  
  return clump

In [ ]:
def newKmeans(dataFileInput, clustNum, centro):

  tVal = 1

  dataFileInput = dataFileInput.sample(frac = 1).reset_index(drop = True)

  if centro == None:

    centro = ogKmeans(dataFileInput, clustNum, centro)
  
  myClust = clump(clustNum, dataFileInput, centro)

  centro2 = newCentro(myClust, clustNum)

  centTweet = list(centro.values())

  newCentTweet = list(centro2.values())

  convg = convgCheck(False, clustNum, centTweet, newCentTweet)
  
  if not convg:

    print("Convergence in Process")

    print("\n")

    centro = centro2.copy()

    newKmeans(dataFileInput, clustNum, centro)

  else:

    print("--------------------------")

    print("\n")

    print("Covergence Completed")

    sum_squared_error_result = sum_squared_error(myClust, centro)

    print("\n")

    print("Sum Squared Error Result Is Shown Below ->")

    print("\n")

    print(sum_squared_error_result)

    for i in range(clustNum):

      clustLen = len(myClust[i])

      clustLenStr = str(clustLen)

      idx_tPos = i + tVal
      
      positionIdx = str(idx_tPos)

      print("\n")

      print("Total Number Of Tweets In Cluster "+ positionIdx + " -> " + clustLenStr)

In [ ]:
def ogKmeans(dataFileInput, clustNum, centro):

  centro = {}

  i = 0

  while(i < clustNum):

    if (dataFileInput[i] not in list(centro.keys())):

      centro[i] = dataFileInput[i]

    i = i + 1
    
  return centro

In [ ]:
def distCalculation(clustnew):
  dist = []

  distSum = 0

  for x in clustnew:

    if x != []:

      distSpan = []

      for y in clustnew:

        distSpan.append(distJaccard(x, y))
        
      distSum = sum(distSpan)
      
      dist.append(distSum)
  
  return dist, distSum

In [ ]:
def newCentro(clump, clustNum):

  centro = {}

  for i in range(clustNum):

    centro[i] = []
  
  for point in clump:

    clustnew = clump[point]
    
    dist, distSum = distCalculation(clustnew)
  
    minDist = min(dist)

    clustIdPlacement = dist.index(minDist)
    
    centro[point] = clustnew[clustIdPlacement]

  return centro

In [ ]:
def convgCheck(convg, clustNum, centTweet, newCentTweet):

  i = 0

  while(i < clustNum):

    if (centTweet[i] != newCentTweet[i]):

      convg = False

      break

    else:

      convg = True
    
    x = i + 1

    i = x
    
  return convg

In [ ]:
def distJaccard(x, y):  

  x = set(x)
  
  y = set(y)

  z = list((x & y))

  zlen = len(z)

  w = list((x | y))

  wlen = len(w)
  
  out = (zlen/wlen)

  return 1 - out

In [ ]:
def sum_squared_error(clump, centro):

  sum = 0

  for placerID in centro.keys():

    for tweets in list(clump[placerID]):

      newJaccard = math.pow(distJaccard(centro[placerID], tweets), 2)

      sum = sum + newJaccard
      
  return sum

In [ ]:
def main():
  
  dataFileInput = 'https://raw.githubusercontent.com/rocktrees/assignment3/main/reuters_health.txt'

  procData = preProc(dataFileInput)

  newKmeans(procData, clustNum = 10, centro = None) 

In [ ]:
main()

806     [astrazeneca's, patent, on, asthma, drug, inva...
514     [treatment, for, prostate, cancer, varies, by,...
930     [moderate, drinking, linked, to, lower, heart,...
3219    [ebola, deaths, reach, 3,338,, but, widely, un...
4069    [patient, in, suspected, ebola, case, in, cana...
                              ...                        
2345    [wockhardt, says, u.s., export, ban, unlikely,...
752            [u.s., cancer, survival, rates, improving]
3828    [heart, doctors, overstate, benefits, of, proc...
4493    [californians, lacking, health, insurance, hal...
4040    [listeria-infected, cold, cuts, kill, 12, in, ...
Name: Input, Length: 4719, dtype: object


Convergence in Process


Convergence in Process


Convergence in Process


--------------------------


Covergence Completed


Sum Squared Error Result Is Shown Below ->


3787.7272291075324


Total Number Of Tweets In Cluster 1 -> 1442


Total Number Of Tweets In Cluster 2 -> 130


Total Number Of Tweets In Cluster 3